In [ ]:
import torch
from PIL import Image
from torch import nn, save, load
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import random

In [ ]:
# Variables
n_epochs = 10  # Number of epochs
lr = 0.001  # Learning rate

In [ ]:
# Load data
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(
    root="../data", download=True, train=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

test_dataset = datasets.MNIST(
    root="../data", download=True, train=False, transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=32, num_workers=2)

In [ ]:
# Define the image classifier model
class ImageClassifier(nn.Module):
    def __init__(self):
        # Call super class initialisation method
        super(ImageClassifier, self).__init__()

        # Define convolution layers
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3),
            nn.ReLU(),
        )

        # Define fully connected layers
        self.fc_layers = nn.Sequential(nn.Flatten(), nn.Linear(64 * 22 * 22, 10))

    def forward(self, x):
        # Forward pass
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

In [ ]:
# Create an instance of the image classifier model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier = ImageClassifier().to(device)

In [ ]:
# Define the optimizer and loss function
optimizer = Adam(classifier.parameters(), lr=lr, weight_decay=0)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Train the model
for epoch in range(n_epochs):
    for images, labels in train_loader:
        # Extract images and labels
        images, labels = images.to(device), labels.to(device)

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = classifier(images)

        # Compute loss
        loss = loss_fn(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

    print(f"Epoch:{epoch} loss is {loss.item()}")

In [ ]:
# Evaluate model performance
classifier.eval()  # Set model to evaluation mode
correct = 0
total = 0

# Disable gradient computation
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = classifier(images)

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
# Pick a random index
index = random.randint(0, len(test_dataset) - 1)
image, label = test_dataset[index]

# Add batch dimension (1, 1, 28, 28)
image = image.unsqueeze(0).to(device)

# Disable gradient computation
with torch.no_grad():
    output = classifier(image)
    _, predicted = torch.max(output, 1)

predicted_label = predicted.item()

# Move image back to CPU for plotting
image = image.squeeze().cpu()

# Plot
plt.imshow(image, cmap="gray")
plt.title(f"Actual: {label} | Predicted: {predicted_label}")
plt.axis("off")
plt.show()